<a href="https://colab.research.google.com/github/zyang91/ai-sustainbility-final/blob/main/work/colab/colab_finetune_log.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune the PaddleOCR recognition model on Penn plan text lines

Runs on a Colab GPU. Before starting: **Runtime > Change runtime type > T4 GPU**.

You need one file from the project: `work/colab/penn_ocr_finetune.zip`
(training and validation line images with human transcriptions, plus the scripts).
The test partition is not in it.

Run the cells top to bottom. The last cell downloads `penn_finetuned.zip`.

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [1]:
import pandas

## 1. Install PaddlePaddle (GPU build, same version as the local baseline)

In [4]:
!pip install paddlepaddle-gpu==3.3.1 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 GB 7.5 MB/s  0:02:54
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 13.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 24.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 14.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 11.4 MB/s  0:00:46
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 9.5 MB/s  0:00:37
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 12.3 MB/s  0:00:16
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 11.1 MB/s  0:00:14
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 11.1 MB/s  0:00:19
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 11.1 MB/s  0:00:14
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.4/201.4 M

In [5]:
!pip install pyclipper lmdb rapidfuzz albumentations \
    scikit-image shapely pyyaml

  Using cached pyclipper-1.4.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (8.6 kB)
  Using cached lmdb-3.0.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (1.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 9.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [lmdb]


In [6]:
import paddle

print("Paddle version:", paddle.__version__)
print("CUDA available:", paddle.device.is_compiled_with_cuda())
print("Device:", paddle.device.get_device())

paddle.utils.run_check()

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Paddle version: 3.3.1
CUDA available: True
Device: gpu:0
Running verify PaddlePaddle program ... 


/usr/local/lib/python3.13/dist-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


## 2. Upload `penn_ocr_finetune.zip`

In [7]:
import os, zipfile
from google.colab import files
os.makedirs("/content/penn", exist_ok=True)
os.chdir("/content/penn")
if not os.path.exists("work/rec_dataset/rec_gt_train.txt"):
    up = files.upload()                       # choose penn_ocr_finetune.zip
    zipfile.ZipFile(next(iter(up))).extractall(".")
print(sum(1 for _ in open("work/rec_dataset/rec_gt_train.txt")), "training lines,",
      sum(1 for _ in open("work/rec_dataset/rec_gt_val.txt")), "validation lines")

Saving penn_ocr_finetune.zip to penn_ocr_finetune.zip
209 training lines, 101 validation lines


## 3. Train

The script clones PaddleOCR v3.7.0, downloads the pretrained PP-OCRv6 weights,
scores the pretrained model on the validation lines, fine-tunes, and scores the
best checkpoint again. Change `--epochs` / `--lr` here if you want another run.

In [8]:
%cd /content/penn
!bash finetune/run_finetune.sh --gpu --epochs 20 --lr 2e-5 --batch-size 8

/content
Note: switching to 'b03f46425e8ff4442b268ce449e3eef758146cd4'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  221M  100  221M    0     0  5253k      0  0:00:43  0:00:43 --:--:-- 15.8M
209 training lines, 26 steps/epoch -> /content/penn/work/finetune/config.yml
== Validation score of the pretrained model, before any training ==
[2026/11/15 19:54:47] ppocr I

## 4. Before / after on the validation lines

In [9]:
import re
for name in ("eval_before", "eval_after"):
    log = open(f"/content/penn/work/finetune/{name}.log").read()
    print(name, {k: float(v) for k, v in re.findall(r"(acc|norm_edit_dis):([0-9.]+)", log)})

eval_before {'acc': 0.950494955396539, 'norm_edit_dis': 0.9964141895910021}
eval_after {'acc': 0.9801979227526809, 'norm_edit_dis': 0.9988696545192166}


## 5. Download the fine-tuned model

In [10]:
%cd /content/penn/work/finetune
!zip -q -r /content/penn_finetuned.zip inference config.yml train.log eval_before.log eval_after.log
files.download("/content/penn_finetuned.zip")

/content/penn/work/finetune


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Back on the Mac, unzip it into `work/finetune/` so that `work/finetune/inference/` exists, then:

```
python -m penn_ocr.predict --engine paddle --rec-model-dir work/finetune/inference --name paddle_finetuned
python -m penn_ocr.evaluate --split val
python -m penn_ocr.evaluate --split test
```